# Deep Learning Interview Preparation

Hands-on PyTorch examples. Install dependencies if needed: `pip install torch scikit-learn matplotlib`.

## 1. Tensors and devices

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4, 10, device=device)
print("shape:", x.shape, "dtype:", x.dtype, "device:", x.device)


## 2. Autograd and the chain rule

PyTorch tracks tensor operations when gradients are enabled. `backward()` computes gradients for leaf tensors that require gradients.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
y.backward()
print("y:", y.item())
print("dy/dx:", x.grad.item())  # 2x + 2, evaluated at x=3


## 3. Define a simple neural network

In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, input_features=10, hidden_features=32, classes=3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, hidden_features),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_features, classes),
        )

    def forward(self, inputs):
        return self.network(inputs)  # raw logits

model = SmallClassifier().to(device)
example = torch.randn(5, 10, device=device)
logits = model(example)
print("logits shape:", logits.shape)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 4. Minimal training loop on synthetic data

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
features = torch.randn(600, 10)
labels = (features[:, 0] + 0.5 * features[:, 1] > 0).long()

dataset = TensorDataset(features, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
binary_model = SmallClassifier(input_features=10, hidden_features=32, classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(binary_model.parameters(), lr=1e-3)

for epoch in range(5):
    binary_model.train()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        predictions = binary_model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
    print(f"epoch={epoch + 1}, loss={total_loss / len(dataset):.4f}")


## 5. Evaluation mode and inference

In [ ]:
binary_model.eval()
with torch.inference_mode():
    sample = torch.randn(8, 10, device=device)
    scores = binary_model(sample)
    predicted_classes = scores.argmax(dim=1)
print(predicted_classes)


## 6. Gradient clipping

In [ ]:
# Typical training-step pattern when gradients may become large:
batch_x, batch_y = next(iter(loader))
batch_x, batch_y = batch_x.to(device), batch_y.to(device)
binary_model.train()
optimizer.zero_grad(set_to_none=True)
loss = criterion(binary_model(batch_x), batch_y)
loss.backward()
torch.nn.utils.clip_grad_norm_(binary_model.parameters(), max_norm=1.0)
optimizer.step()
print("loss:", loss.item())


## 7. Convolutional neural network

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),  # assumes 28x28 input
            nn.ReLU(),
            nn.Linear(64, classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))

cnn = SmallCNN().to(device)
print("CNN output shape:", cnn(torch.randn(4, 1, 28, 28, device=device)).shape)


## 8. Scaled dot-product attention

In [ ]:
import math

def scaled_dot_product_attention(query, key, value, mask=None):
    scores = query @ key.transpose(-2, -1) / math.sqrt(query.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ value, weights

q = torch.randn(2, 4, 8, device=device)
k = torch.randn(2, 6, 8, device=device)
v = torch.randn(2, 6, 12, device=device)
context, weights = scaled_dot_product_attention(q, k, v)
print("context:", context.shape)  # batch, query length, value dimension
print("weights:", weights.shape)  # batch, query length, key length


## 9. Save and load model weights

In [ ]:
# Save weights (state dictionary), not an arbitrary serialized model object.
checkpoint_path = "small_classifier_state.pt"
torch.save(binary_model.state_dict(), checkpoint_path)

restored = SmallClassifier(input_features=10, hidden_features=32, classes=2)
state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored.load_state_dict(state)
restored.eval()
print("Loaded model with", sum(p.numel() for p in restored.parameters()), "parameters")


## Interview prompts

1. Why should `CrossEntropyLoss` normally receive raw logits?
2. What is the difference between `model.eval()` and `torch.inference_mode()`?
3. How would you diagnose a loss that becomes NaN?
4. Why does standard self-attention memory use grow quickly with sequence length?
5. How would you tell overfitting apart from a train/serve distribution mismatch?